In [1]:
"""
Cricsheet IPL 2026 Data Loader
===============================
Loads IPL 2026 season data from a full Cricsheet IPL archive folder
and produces 3 clean CSV files:
 
    players.csv    — one row per player (name + Cricsheet ID)
    matches.csv    — one row per match (venue, teams, result etc.)
    deliveries.csv — one row per ball bowled
 
Only matches with ID >= 1527674 are loaded (start of 2026 IPL season).
 
Usage:
    1. Set DATA_FOLDER to the path of your folder
    2. Run: python cricket_loader.py
"""

'\nCricsheet IPL 2026 Data Loader\n===============================\nLoads IPL 2026 season data from a full Cricsheet IPL archive folder\nand produces 3 clean CSV files:\n\n    players.csv    — one row per player (name + Cricsheet ID)\n    matches.csv    — one row per match (venue, teams, result etc.)\n    deliveries.csv — one row per ball bowled\n\nOnly matches with ID >= 1527674 are loaded (start of 2026 IPL season).\n\nUsage:\n    1. Set DATA_FOLDER to the path of your folder\n    2. Run: python cricket_loader.py\n'

### Import Python libraries and initialize constants

In [2]:
import pandas as pd
import os

IPL_DATA_FOLDER = "./ipl_male_csv2"
FIRST_2026_MATCH = 1527674
#pd.set_option('display.max_rows', None)

### Get 2026 Match IDs

In [3]:
def get_2026_match_IDs(folder_path):
    """
    Scans the folder for all delivery CSVs and returns only
    match IDs >= FIRST_2026_MATCH.
    """
    all_ids = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".csv") and not filename.endswith("_info.csv"):
            try:
                match_id = int(filename.replace(".csv", ""))
                if match_id >= FIRST_2026_MATCH:
                    all_ids.append(match_id)
            except ValueError:
                pass  # skip any files that don't have a numeric name
 
    all_ids.sort()
    return all_ids

### Get 2026 Match Info and Player files

In [4]:
# -------------------------------------------------------
# STEP 2: Load info files → matches.csv + players.csv
# -------------------------------------------------------
def load_info_files(folder, match_ids):
    """
    Reads _info.csv for each 2026 match and returns:
      - matches_df : one row per match
      - players_df : unique player registry (name + Cricsheet ID)
    """
    all_matches = []
    all_registry = []
    all_players = []
 
    for match_id in match_ids:
        filename = f"{match_id}_info.csv"
        filepath = os.path.join(folder, filename)
        if not os.path.exists(filepath):
            print(f"  Warning: {filename} not found, skipping")
            continue
 
        try:
            df = pd.read_csv(filepath, header=None, skiprows=[0], engine='python', names=range(5))
 
            # Keep only info rows
            df = df[df[0] == "info"].copy()
            #print(df.to_string())
            # --- People registry rows ---
            # Format: info, registry, people, Name, cricsheet_id
            registry_rows = df[df[1] == "registry"]
            #print(registry_rows)
            for _, row in registry_rows.iterrows():
                name  = row[3] if pd.notna(row[3]) else None
                cricsheet_id = row[4] if len(row) > 4 and pd.notna(row[4]) else None
                if name and cricsheet_id:
                    all_registry.append({
                        "player_name":  name,
                        "cricsheet_id": cricsheet_id
                    })
 
            # --- Match info rows (everything except registry and players) ---
            info_rows = df[~df[1].isin(["registry", "player"])]
            match_dict = {"match_id": match_id}
 
            for _, row in info_rows.iterrows():
                key   = row[1]
                value = "|".join(
                    str(v) for v in row[2:] if pd.notna(v)
                )
                # If key seen before, append with pipe separator
                if key in match_dict:
                    match_dict[key] = match_dict[key] + "|" + value
                else:
                    match_dict[key] = value
 
            all_matches.append(match_dict)
 
            # --- Player info rows (everything except registry and players) ---
            player_team_rows = df[(df[1] == "player")]
            #print(player_team_rows.to_string())
            for _, row in player_team_rows.iterrows():
                player_name = row[3]
                player_team = row[2]
                if player_name and player_team:
                    all_players.append({
                        "player_name": player_name,
                        "team": player_team

                    })



        except Exception as e:
            print(f"  Warning: could not load {filename} — {e}")
 
    matches_df = (pd.DataFrame(all_matches))
    #print(matches_df)
    registry_df = (pd.DataFrame(all_registry).drop_duplicates(subset="player_name"))
    #print(registry_df)                
    team_df = (pd.DataFrame(all_players).drop_duplicates(subset="player_name"))
    #print(team_df)
    players_df = (pd.merge(registry_df, team_df, on="player_name", how='left').sort_values("team").reset_index(drop=True))
 
    print(f"✓ Matches loaded:  {len(matches_df)}")
    print(f"✓ Players found:   {len(players_df)}")
    return matches_df, players_df

In [5]:
def load_delivery_files(folder, match_ids):
    """
    Reads the ball-by-ball delivery CSV for each 2026 match.
    
    """
    all_deliveries = []
 
    #print(f"\nLoading delivery files...")
 
    for match_id in match_ids:
        filename = f"{match_id}.csv"
        filepath = os.path.join(folder, filename)
 
        if not os.path.exists(filepath):
            print(f"  Warning: {filename} not found, skipping")
            continue
 
        try:
            df = pd.read_csv(filepath)
            #df.insert(0, "match_id", match_id)  # add match_id as first column
            all_deliveries.append(df)
 
        except Exception as e:
            print(f"  Warning: could not load {filename} — {e}")
 
    deliveries_df = pd.concat(all_deliveries, ignore_index=True)
    print(f"✓ Total deliveries loaded: {len(deliveries_df):,}")
    return deliveries_df

In [6]:
match_ids = get_2026_match_IDs(IPL_DATA_FOLDER)
match_ids

[1527674,
 1527675,
 1527676,
 1527677,
 1527678,
 1527679,
 1527680,
 1527681,
 1527682,
 1527683,
 1527684,
 1527685,
 1527686,
 1527687,
 1527688,
 1527689,
 1527690,
 1527691,
 1527692,
 1527693,
 1529264,
 1529265,
 1529266,
 1529267,
 1529268,
 1529269,
 1529270,
 1529271,
 1529272,
 1529273,
 1529274,
 1529275,
 1529276,
 1529277,
 1529278,
 1529279,
 1529280,
 1529281,
 1529282,
 1529283,
 1529284,
 1529285,
 1529286,
 1529287,
 1529288,
 1529289,
 1529290,
 1529291,
 1529292,
 1529293,
 1529294,
 1529295,
 1529296,
 1529297,
 1529298,
 1529299,
 1529300,
 1529301,
 1529302,
 1529303,
 1529304,
 1529305,
 1529306,
 1529307,
 1529308,
 1529309]

In [7]:
display(len(match_ids))

66

In [8]:
matches_df, players_df= load_info_files(IPL_DATA_FOLDER, match_ids)
matches_df

✓ Matches loaded:  66
✓ Players found:   237


,match_id,balls_per_over,team,gender,season,date,event,match_number,venue,city,...,umpire,reserve_umpire,tv_umpire,match_referee,winner,winner_wickets,winner_runs,outcome,eliminator,method
0,1527674,6,Sunrisers Hyderabad|Royal Challengers Bengaluru,male,2026,2026/03/28,Indian Premier League,1,"M Chinnaswamy Stadium, Bengaluru",Bengaluru,...,J Madanagopal|UV Gandhe,A Bengeri,R Pandit,J Srinath,Royal Challengers Bengaluru,6,NaN,NaN,NaN,NaN
1,1527675,6,Kolkata Knight Riders|Mumbai Indians,male,2026,2026/03/29,Indian Premier League,2,"Wankhede Stadium, Mumbai",Mumbai,...,Anish Sahasrabudhe|MV Saidharshan Kumar,Vinod Seshan,Nitin Menon,Shakti Singh,Mumbai Indians,6,NaN,NaN,NaN,NaN
2,1527676,6,Chennai Super Kings|Rajasthan Royals,male,2026,2026/03/30,Indian Premier League,3,"Barsapara Cricket Stadium, Guwahati",Guwahati,...,HDPK Dharmasena|M Krishnadas,K Swaroopanand,VK Sharma,V Narayan Kutty,Rajasthan Royals,8,NaN,NaN,NaN,NaN
3,1527677,6,Gujarat Titans|Punjab Kings,male,2026,2026/03/31,Indian Premier League,4,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,...,Abhijit Bhattacharya|A Totre,P Joshi,KN Ananthapadmanabhan,Prakash Bhatt,Punjab Kings,3,NaN,NaN,NaN,NaN
4,1527678,6,Lucknow Super Giants|Delhi Capitals,male,2026,2026/04/01,Indian Premier League,5,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,...,R Pandit|UV Gandhe,A Bengeri,J Madanagopal,J Srinath,Delhi Capitals,6,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,1529305,6,Rajasthan Royals|Delhi Capitals,male,2026,2026/05/17,Indian Premier League,62,"Arun Jaitley Stadium, Delhi",Delhi,...,Amit Rana|K Kelkar,T Ematty,J Madanagopal,DS Manohar,Delhi Capitals,5,NaN,NaN,NaN,NaN
62,1529306,6,Chennai Super Kings|Sunrisers Hyderabad,male,2026,2026/05/18,Indian Premier League,63,"MA Chidambaram Stadium, Chepauk, Chennai",Chennai,...,Nitin Menon|TM Srivastava,KM Gandhi,Anish Sahasrabudhe,A Kripal Singh,Sunrisers Hyderabad,5,NaN,NaN,NaN,NaN
63,1529307,6,Lucknow Super Giants|Rajasthan Royals,male,2026,2026/05/19,Indian Premier League,64,"Sawai Mansingh Stadium, Jaipur",Jaipur,...,AT Holdstock|P Joshi,A Totre,Bhavesh Patel,IR Siddiqui,Rajasthan Royals,7,NaN,NaN,NaN,NaN
64,1529308,6,Mumbai Indians|Kolkata Knight Riders,male,2026,2026/05/20,Indian Premier League,65,"Eden Gardens, Kolkata",Kolkata,...,K Swaroopanand|HAS Khalid,AK Argal,M Krishnadas,R Seth,Kolkata Knight Riders,4,NaN,NaN,NaN,NaN


In [9]:
players_df

,player_name,cricsheet_id,team
0,A Kamboj,fcc21ace,Chennai Super Kings
1,A Mhatre,b2b4f545,Chennai Super Kings
2,J Overton,59559bc2,Chennai Super Kings
3,KK Ahmed,a2f46292,Chennai Super Kings
4,Kartik Sharma,7b679de5,Chennai Super Kings
...,...,...,...
232,P Dharmani,eaa90ab4,NaN
233,T Ematty,e995f8e9,NaN
234,DS Manohar,b35f3ac8,NaN
235,P Dubey,ed5a5510,NaN


In [10]:
deliveries_df = load_delivery_files(IPL_DATA_FOLDER, match_ids)
deliveries_df

✓ Total deliveries loaded: 15,559


,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,...,extras,wides,noballs,byes,legbyes,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed
0,1527674,2026,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",1,0.1,Sunrisers Hyderabad,Royal Challengers Bengaluru,TM Head,Abhishek Sharma,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1527674,2026,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",1,0.2,Sunrisers Hyderabad,Royal Challengers Bengaluru,TM Head,Abhishek Sharma,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1527674,2026,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",1,0.3,Sunrisers Hyderabad,Royal Challengers Bengaluru,Abhishek Sharma,TM Head,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1527674,2026,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",1,0.4,Sunrisers Hyderabad,Royal Challengers Bengaluru,Abhishek Sharma,TM Head,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1527674,2026,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",1,0.5,Sunrisers Hyderabad,Royal Challengers Bengaluru,Abhishek Sharma,TM Head,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15554,1529309,2026,2026-05-21,"Narendra Modi Stadium, Ahmedabad",2,12.7,Chennai Super Kings,Gujarat Titans,Mukesh Choudhary,Noor Ahmad,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15555,1529309,2026,2026-05-21,"Narendra Modi Stadium, Ahmedabad",2,13.1,Chennai Super Kings,Gujarat Titans,Noor Ahmad,Mukesh Choudhary,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15556,1529309,2026,2026-05-21,"Narendra Modi Stadium, Ahmedabad",2,13.2,Chennai Super Kings,Gujarat Titans,Noor Ahmad,Mukesh Choudhary,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15557,1529309,2026,2026-05-21,"Narendra Modi Stadium, Ahmedabad",2,13.3,Chennai Super Kings,Gujarat Titans,Noor Ahmad,Mukesh Choudhary,...,0,NaN,NaN,NaN,NaN,NaN,caught and bowled,Noor Ahmad,NaN,NaN
